In [ ]:
import sqlite3
from pathlib import Path

database_path = Path("../data/db/edition.sqlite")

def run_query(query: str, params: tuple = ()):
    conn = sqlite3.connect(database_path)
    cur = conn.cursor()
    cur.execute(query, params)
    cols = [d[0] for d in cur.description] if cur.description else []
    rows = cur.fetchall()
    conn.close()
    return cols, rows

def pretty_print(cols, rows, max_rows=20):
    if not rows:
        print("(no rows)")
        return
    print(" | ".join(cols))
    print("-" * 80)
    for r in rows[:max_rows]:
        print(" | ".join("" if v is None else str(v) for v in r))
    if len(rows) > max_rows:
        print(f"... ({len(rows)-max_rows} more rows)")


In [ ]:
queries = {
    "S1_schema": {
        "title": "Strukturelle Exploration (Schema)",
        "master": """
            SELECT type, name
            FROM sqlite_master
            WHERE type IN ('table','view')
              AND name NOT LIKE 'sqlite_%'
            ORDER BY type, name;
        """,
        "support": {
            "columns_site": "PRAGMA table_info(site);",
            "fk_site_map": "PRAGMA foreign_key_list(site_map);",
        },
        "narrative": [
            "Ziel: Überblick über Tabellen/Views als Datenmodell der Extraktion.",
            "Master zeigt: welche Entitäten und Relationen überhaupt existieren."
        ]
    },

    "S2_quality": {
        "title": "Vollständigkeit & Datenqualität",
        "master": """
            SELECT
              COUNT(*) AS n_sites,
              SUM(CASE WHEN utm_easting  IS NULL THEN 1 ELSE 0 END) AS n_null_easting,
              SUM(CASE WHEN utm_northing IS NULL THEN 1 ELSE 0 END) AS n_null_northing,
              ROUND(100.0 * SUM(CASE WHEN utm_easting IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_easting,
              ROUND(100.0 * SUM(CASE WHEN utm_northing IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null_northing
            FROM site;
        """,
        "support": {
            "dup_site_codes": """
                SELECT code, COUNT(*) AS n
                FROM site
                GROUP BY code
                HAVING COUNT(*) > 1
                ORDER BY n DESC, code;
            """,
            "orphans_coords": """
                SELECT c.*
                FROM coords c
                LEFT JOIN site s ON s.id = c.site_id
                WHERE s.id IS NULL;
            """,
            "text_len_finds": """
                SELECT
                  COUNT(*) AS n_sites,
                  SUM(CASE WHEN finds_text IS NULL OR TRIM(finds_text) = '' THEN 1 ELSE 0 END) AS n_empty,
                  ROUND(100.0 * SUM(CASE WHEN finds_text IS NULL OR TRIM(finds_text) = '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_empty,
                  MIN(LENGTH(finds_text)) AS min_len,
                  MAX(LENGTH(finds_text)) AS max_len,
                  ROUND(AVG(LENGTH(finds_text)), 2) AS avg_len
                FROM site;
            """
        },
        "narrative": [
            "Ziel: Einschätzung, welche Analysen robust sind (Missingness, Orphans, Dubletten).",
            "Master zeigt: Koordinaten-Vollständigkeit als Schlüsselvariable für räumliche Analysen."
        ]
    },

    "S3_spatial": {
        "title": "Räumliche Exploration (Sites & Koordinaten)",
        "master": """
            SELECT
              COUNT(*) AS n_sites_total,
              SUM(CASE WHEN c.site_id IS NOT NULL THEN 1 ELSE 0 END) AS n_sites_with_coords,
              ROUND(100.0 * SUM(CASE WHEN c.site_id IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_sites_with_coords
            FROM site s
            LEFT JOIN coords c ON c.site_id = s.id;
        """,
        "support": {
            "bbox": """
                SELECT
                  MIN(latitude)  AS min_lat,
                  MAX(latitude)  AS max_lat,
                  MIN(longitude) AS min_lon,
                  MAX(longitude) AS max_lon,
                  COUNT(*) AS n_points
                FROM coords
                WHERE latitude IS NOT NULL
                  AND longitude IS NOT NULL;
            """,
            "site_export": """
                SELECT
                  s.id,
                  s.code AS site_code,
                  s.name AS site_name,
                  v.name AS village_name,
                  c.latitude,
                  c.longitude
                FROM site s
                JOIN village v ON v.id = s.village_id
                LEFT JOIN coords c ON c.site_id = s.id
                ORDER BY v.name, s.code;
            """,
        },
        "narrative": [
            "Ziel: Prüfen, ob GIS/Mapping sinnvoll ist (Coverage + plausible ranges).",
            "Master zeigt: Anteil lokalisierter Sites."
        ]
    },

    "S4_chrono": {
        "title": "Chronologische Exploration (Perioden/Phasen)",
        "master": """
            SELECT period, COUNT(*) AS n_classes
            FROM pottery_class
            GROUP BY period
            ORDER BY n_classes DESC, period;
        """,
        "support": {
            "find_item_periods": """
                SELECT period, COUNT(*) AS n_items
                FROM find_item
                GROUP BY period
                ORDER BY n_items DESC, period;
            """,
            "multi_period_sites": """
                SELECT
                  pcs.site_id,
                  s.code AS site_code,
                  COUNT(DISTINCT pc.period) AS n_periods,
                  GROUP_CONCAT(DISTINCT pc.period) AS periods
                FROM pottery_class_site pcs
                JOIN pottery_class pc ON pc.id = pcs.pottery_class_id
                JOIN site s ON s.id = pcs.site_id
                GROUP BY pcs.site_id, s.code
                HAVING COUNT(DISTINCT pc.period) > 1
                ORDER BY n_periods DESC, s.code;
            """,
        },
        "narrative": [
            "Ziel: Perioden-Verteilung + Mehrperiodigkeit als Basis für Trendanalysen.",
            "Master zeigt: Schwerpunktsetzung (Dissertations-/Publikationsbias vs. archäologische Realität)."
        ]
    },

    "S5_context": {
        "title": "Kontextuelle Exploration (Relations: Sites × Maps/Plates/Figures)",
        "master": """
            WITH m AS (
              SELECT site_id, COUNT(*) AS n_maps
              FROM site_map
              GROUP BY site_id
            ),
            p AS (
              SELECT site_id, COUNT(*) AS n_plates
              FROM site_plate
              GROUP BY site_id
            ),
            f AS (
              SELECT site_id, COUNT(*) AS n_figures
              FROM site_figure
              GROUP BY site_id
            )
            SELECT
              s.code AS site_code,
              COALESCE(m.n_maps, 0)   AS n_maps,
              COALESCE(p.n_plates, 0) AS n_plates,
              COALESCE(f.n_figures, 0) AS n_figures,
              (COALESCE(m.n_maps, 0) + COALESCE(p.n_plates, 0) + COALESCE(f.n_figures, 0)) AS doc_score
            FROM site s
            LEFT JOIN m ON m.site_id = s.id
            LEFT JOIN p ON p.site_id = s.id
            LEFT JOIN f ON f.site_id = s.id
            ORDER BY doc_score DESC, s.code
            LIMIT 25;
        """,
        "support": {
            "maps_per_site": """
                SELECT s.code AS site_code, COUNT(sm.map_id) AS n_maps
                FROM site s
                LEFT JOIN site_map sm ON sm.site_id = s.id
                GROUP BY s.code
                ORDER BY n_maps DESC, s.code;
            """,
            "plates_per_site": """
                SELECT s.code AS site_code, COUNT(sp.plate_id) AS n_plates
                FROM site s
                LEFT JOIN site_plate sp ON sp.site_id = s.id
                GROUP BY s.code
                ORDER BY n_plates DESC, s.code;
            """,
            "figures_per_site": """
                SELECT s.code AS site_code, COUNT(sf.figure_id) AS n_figures
                FROM site s
                LEFT JOIN site_figure sf ON sf.site_id = s.id
                GROUP BY s.code
                ORDER BY n_figures DESC, s.code;
            """,
        },
        "narrative": [
            "Ziel: Identifikation 'Key Sites' für Fallstudien und Bias-Kontrolle.",
            "Master zeigt: Top 25 nach Dokumentationsdichte (ein sehr präsentationsstarker Output)."
        ]
    },

    "S6_extraction": {
        "title": "Methodische Exploration der Extraktion (Quellen, Textlastigkeit)",
        "master": """
            SELECT source_file, COUNT(*) AS n
            FROM pottery_class
            GROUP BY source_file
            ORDER BY n DESC, source_file;
        """,
        "support": {
            "find_item_sources": """
                SELECT source_file, COUNT(*) AS n
                FROM find_item
                GROUP BY source_file
                ORDER BY n DESC, source_file;
            """,
            "text_instead_of_relations": """
                SELECT
                  COUNT(*) AS n_classes,
                  SUM(CASE WHEN sites_text   IS NOT NULL AND TRIM(sites_text)   <> '' THEN 1 ELSE 0 END) AS n_sites_text_filled,
                  SUM(CASE WHEN figures_text IS NOT NULL AND TRIM(figures_text) <> '' THEN 1 ELSE 0 END) AS n_figures_text_filled
                FROM pottery_class;
            """,
        },
        "narrative": [
            "Ziel: Transparenz über Datenentstehung und Extraktionsartefakte.",
            "Master zeigt: welche Quellsegmente den Datensatz dominieren."
        ]
    },

    "S7_hypotheses": {
        "title": "Hypothesen-getriebene Mini-Explorationen",
        "master": """
            SELECT
              pc.period,
              SUM(COALESCE(pcs.sherd_count, 0)) AS total_sherds,
              COUNT(DISTINCT pcs.site_id) AS n_sites
            FROM pottery_class_site pcs
            JOIN pottery_class pc ON pc.id = pcs.pottery_class_id
            GROUP BY pc.period
            ORDER BY total_sherds DESC, pc.period;
        """,
        "support": {
            "sampling_intensity_text": """
                SELECT
                  s.code AS site_code,
                  LENGTH(COALESCE(pcs.sample_codes_text, '')) AS sample_codes_text_len,
                  pcs.sample_codes_text
                FROM pottery_class_site pcs
                JOIN site s ON s.id = pcs.site_id
                WHERE pcs.sample_codes_text IS NOT NULL
                  AND TRIM(pcs.sample_codes_text) <> ''
                ORDER BY sample_codes_text_len DESC
                LIMIT 50;
            """,
        },
        "narrative": [
            "Ziel: erste, demonstrative 'Settlement Trend'-Indikatoren aus den Daten.",
            "Master zeigt: Periodenweise Summen (Sherds) + Anzahl Sites als grobe Intensitätsmetrik."
        ]
    }
}


In [ ]:
def run_master(scenario_key: str, max_rows=30):
    s = queries[scenario_key]
    print(f"=== {scenario_key}: {s['title']} ===")
    for line in s.get("narrative", []):
        print(f"- {line}")
    cols, rows = run_query(s["master"])
    pretty_print(cols, rows, max_rows=max_rows)


In [ ]:
def run_support(scenario_key: str, support_key: str, params: tuple = (), max_rows=30):
    s = queries[scenario_key]
    q = s["support"][support_key]
    print(f"--- {scenario_key} / support: {support_key} ---")
    cols, rows = run_query(q, params=params)
    pretty_print(cols, rows, max_rows=max_rows)


In [ ]:
run_master("S2_quality")
run_support("S2_quality", "dup_site_codes")


=== S2_quality: Vollständigkeit & Datenqualität ===
- Ziel: Einschätzung, welche Analysen robust sind (Missingness, Orphans, Dubletten).
- Master zeigt: Koordinaten-Vollständigkeit als Schlüsselvariable für räumliche Analysen.
n_sites | n_null_easting | n_null_northing | pct_null_easting | pct_null_northing
--------------------------------------------------------------------------------
144 | 3 | 3 | 2.08 | 2.08
--- S2_quality / support: dup_site_codes ---
(no rows)


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

database_path = Path("../data/db/edition.sqlite")

q = """
SELECT
  s.id AS site_id,
  s.code AS site_code,
  s.name AS site_name,
  v.name AS village_name,
  c.latitude,
  c.longitude
FROM coords c
JOIN site s ON s.id = c.site_id
LEFT JOIN village v ON v.id = s.village_id
WHERE c.latitude IS NOT NULL
  AND c.longitude IS NOT NULL
  AND c.latitude BETWEEN -90 AND 90
  AND c.longitude BETWEEN -180 AND 180;
"""

with sqlite3.connect(database_path) as conn:
    df = pd.read_sql_query(q, conn)

df.head()


,site_id,site_code,site_name,village_name,latitude,longitude
0,1,CS.1.1,AI-Fulayj CS.1.1,AL-FULAYJ VILLAGE,22.888825,58.061360
1,2,CS.1.2,AI-Fulayj CS.1.2,AL-FULAYJ VILLAGE,22.882912,58.052745
2,3,CS.1.3,AI-Fulayj CS.1.3,AL-FULAYJ VILLAGE,22.874323,58.051197
3,4,CS.1.4,AI-Fulayj CS.1.4,AL-FULAYJ VILLAGE,22.866359,58.050853
4,5,CS.1.5,AI-Fulayj CS.1.5,AL-FULAYJ VILLAGE,22.859912,58.781993


In [ ]:
import folium

# Kartenmittelpunkt automatisch aus Daten:
center = [df["latitude"].mean(), df["longitude"].mean()]

m = folium.Map(
    location=center,
    zoom_start=8,
    tiles="OpenStreetMap"
)

for _, row in df.iterrows():
    popup_html = f"""
    <b>{row['site_code']}</b><br>
    {row['site_name'] or ''}<br>
    Village: {row['village_name'] or ''}
    """
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        fill=True,
        popup=folium.Popup(popup_html, max_width=320),
        tooltip=row["site_code"],
    ).add_to(m)

m
